# BFS at two extremes: GPU throughput and microsecond latency

Agent traversals come in two shapes: *"sweep the whole graph"* (reachability,
levels, components) and *"expand this tiny neighborhood right now"* (memory
lookups, ego contexts). metal-graph handles both with one API:

- big traversals run **GPU-resident** (whole levels per command buffer,
  direction-optimizing bottom-up switch);
- tiny reachable components are detected by a bounded preflight and answered
  by a serial CPU path in **microseconds** — telemetry tells you which one
  actually ran.

Graph: 2M-edge power-law blob **plus a 40-vertex ring** that is deliberately
disconnected — our stand-in for a small "episode" cluster in an agent
memory.

In [1]:
import time
import numpy as np
import metal_graph as mg

rng = np.random.default_rng(42)
V_blob, E = 200_000, 2_000_000
src = (rng.zipf(1.3, E) - 1) % V_blob
dst = (rng.zipf(1.3, E) - 1) % V_blob
# append the disconnected 40-vertex ring: ids V_blob .. V_blob+39
ring = np.arange(40, dtype=np.uint32) + V_blob
src = np.concatenate([src.astype(np.uint32), ring])
dst = np.concatenate([dst.astype(np.uint32),
                      np.roll(ring, -1)])
V = V_blob + 40

t0 = time.perf_counter()
G = mg.Graph.from_edges(src, dst, directed=True, num_vertices=V)
print(f"built V={G.num_vertices:,} E={G.num_edges:,} in "
      f"{(time.perf_counter()-t0)*1e3:.0f} ms")

built V=200,040 E=2,000,040 in 241 ms


## Full-graph BFS from the super-hub (GPU)

In [2]:
hub = int(np.bincount(src[:E], minlength=V).argmax())
mg.bfs(G, sources=[hub], direction="out")  # warm-up
t0 = time.perf_counter()
dist, parent = mg.bfs(G, sources=[hub], direction="out")
ms = (time.perf_counter() - t0) * 1e3
d = np.asarray(dist)
info = mg.last_run_info()
print(f"BFS from hub {hub}: {ms:.2f} ms | path={info['path']} "
      f"op={info['op']} levels={info['iterations']}")
print(f"reached {(d >= 0).sum():,} of {V:,} vertices")

BFS from hub 0: 4.56 ms | path=gpu op=bfs levels=5
reached 80,202 of 200,040 vertices


## The same call on a tiny component costs microseconds

BFS from a ring vertex can only ever reach 40 vertices. The planner's
bounded preflight notices and answers on the CPU without touching the GPU —
`op` flips to `bfs_sparse`. Forcing `mode="gpu"` shows what that decision
is worth.

In [3]:
ring_start = V_blob  # first ring vertex

def med_us(f, n=200):
    ts = []
    for _ in range(n):
        t0 = time.perf_counter(); f(); ts.append((time.perf_counter()-t0)*1e6)
    return float(np.median(ts))

auto_us = med_us(lambda: mg.bfs(G, sources=[ring_start], direction="out"))
info = mg.last_run_info()
print(f"auto planner : {auto_us:7.1f} us | op={info['op']} "
      f"path={info['path']}")

sparse_us = med_us(lambda: mg.bfs(G, sources=[ring_start], direction="out",
                                  output="sparse"))
print(f"sparse output: {sparse_us:7.1f} us | O(|reached|) result arrays")

if mg.has_gpu():
    mg.set_execution("gpu")
    mg.bfs(G, sources=[ring_start], direction="out")
    gpu_us = med_us(lambda: mg.bfs(G, sources=[ring_start],
                                   direction="out"), n=30)
    mg.set_execution("auto")
    print(f"forced GPU   : {gpu_us:7.1f} us | full dispatch pipeline "
          f"({gpu_us/auto_us:.0f}x the planner's choice)")

auto planner :    15.4 us | op=bfs_sparse path=cpu
sparse output:     5.7 us | O(|reached|) result arrays


forced GPU   :  2318.2 us | full dispatch pipeline (150x the planner's choice)


In [4]:
vs, dv, pv = mg.bfs(G, sources=[ring_start], direction="out",
                    output="sparse")
print("sparse result: vertices", np.asarray(vs).size,
      "| depths 0..", int(np.asarray(dv).max()))
print("first five:", list(zip(np.asarray(vs)[:5].tolist(),
                              np.asarray(dv)[:5].tolist())))

sparse result: vertices 40 | depths 0.. 39
first five: [(200000, 0), (200001, 1), (200002, 2), (200003, 3), (200004, 4)]


## Ego extraction and components on the same snapshot

In [5]:
t0 = time.perf_counter()
vs, es = mg.k_hop(G, seeds=[hub], k=1, direction="out",
                  max_vertices=5_000, max_edges=20_000)
print(f"capped 1-hop ego of the hub: {len(vs):,} vertices, "
      f"{len(es):,} edges in {(time.perf_counter()-t0)*1e3:.1f} ms "
      f"(caps keep agent latency predictable)")

t0 = time.perf_counter()
comp = np.asarray(mg.experimental.wcc(G))
print(f"WCC: {int(comp.max())+1:,} components in "
      f"{(time.perf_counter()-t0)*1e3:.1f} ms | "
      f"ring is its own component: "
      f"{len(set(comp[V_blob:].tolist())) == 1 and comp[V_blob] != comp[hub]}")

capped 1-hop ego of the hub: 5,000 vertices, 20,000 edges in 31.9 ms (caps keep agent latency predictable)
WCC: 79,772 components in 13.7 ms | ring is its own component: True


**Takeaways** — one API, two regimes: multi-million-edge sweeps in
milliseconds on the GPU, tiny-neighborhood answers in microseconds on the
bounded CPU path, chosen automatically and reported honestly. Caps make
worst-case latency predictable for agent loops.